# Lab 07: Archived Web Pages and the Wayback Machine

*INFO 4617/5617 · Week 7 · Chapter 7, [Archived Web Pages and the Wayback Machine](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-07-archives.html)*

This week every class is live coding: one lab in three parts, one a day. The case is the chapter's: Facebook's Terms of Service, `facebook.com/terms`, with stops at Thefacebook in 2004 and CNN in 2000. On Monday, we read the Wayback Machine's addresses by hand, and ask its **Availability API** for one capture. On Wednesday, its **CDX Server** lists every capture of a page, and what the crawler was told each time. On Friday, we fetch ten versions of the terms and measure how much each changed, with a **bag of words**.

Each code cell holds a comment that says what to write; cells marked `# Given` are already written, and we read and run them together. After each step, a **💬 Discuss** box holds questions to talk over; click it to open it. Short ⏱ microsprints ask you to change one thing, alone or in pairs. Nothing here is turned in.

**Before class:** open this notebook in the `webdata` environment (chapter 1), and have your email address ready for the User-Agent. Check that `import gensim` runs; if it doesn't, the chapter's [Document Similarity with Bag-of-Words](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-07-archives.html#document-similarity-with-bag-of-words) gives the one `conda install` line. You need no key and no account. The Wayback Machine limits how many requests one network address may send, and everyone in this room shares one, so run each cell once, when we do. If a cell stops with an error, read it: the Archive may have turned us away. Where a step has a saved copy, the cell under it says how to load it.

On Wednesday and Friday, start at that day's setup step.

> **Instructor key.** Rebuilt on 2026-11-17 for the next offering: three parts, each from its own setup step, with a small first call before each workflow (`docs/plans/2026-11-16-incremental-style-revisions.md`, "Week 07"). It follows chapter 7 at textbook commit `3f9cf2c`. **Most of it has not run.** `web.archive.org`, which serves the captures and the CDX Server, reset every connection from the course's cloud machine for the 30 minutes it was given that day, as it did on 2026-10-21. So 30 code cells are tagged `not-run-live` and have no outputs: Step 1's two redirect cells and its microsprint; Step 2's two requests and its comparison; Step 5 from the capture's fetch on; Steps 6 to 9, but for their given parameters; Step 11 from the page's fetch on; and Step 13. Their code was run once against a stand-in for `web.archive.org` that returned made-up pages and rows, to check that each cell does what its prompt says; none of that output is in this file. What did run, each part in a fresh kernel: every request to the Availability API (`archive.org`, which answers this machine), and all of Step 12. The Availability API's answers moved during the day: it named `www.cnn.com`'s capture of August 15, 2000 in the morning and again in the run kept here, and for a while in between answered the same request with status 200, an empty `archived_snapshots` and an `"error"` of `timeout`; and after about 35 requests in ten minutes it answered `429 Too Many Requests`, as HTML. Before class, run the key from a laptop with `labs/tools/run_key.py`, make the four checkpoints (`labs/week-07/data/README.md` says how; none exists yet), then rewrite this note and Step 14's answers 1 and 2, and check the Discuss questions that cite 2026-11-17 (Steps 3, 5 and 11). The CDX queries in Steps 7 and 9 end at past dates, so their counts should match a later rerun unless the Archive adds or withdraws captures; Steps 5 and 11 depend on which capture the Availability API names, which can change.

# Part 1 — Monday: the archive by hand, then one capture from its API

A capture's address is a query you can read and rewrite. We take one apart, see what the Wayback Machine adds to a page when it replays it, and then put a small question to its Availability API: one whose answer we already know.

## Step 0 — Set up (3 minutes)

Put your email in the given cell's User-Agent, in the book's form, and run it. Every request this week is a plain [`requests.get()`](https://requests.readthedocs.io/en/stable/api/#requests.get) with `headers=HEADERS`, as in week 1. The Internet Archive is a nonprofit library that serves more than a trillion pages for free. When one address sends too much, it answers [`429 Too Many Requests`](https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Status/429), or drops the connection.

In [1]:
# Given: imports, and your User-Agent (put your email in it)
import requests
import pandas as pd
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "WebDataScience/1.0 (your-email@colorado.edu)"}

<details>
<summary>💬 Discuss</summary>

- The Archive limits requests per network address, and this room shares one. What should thirty people running the same cell do differently from one person running it at home?
- Week 2 suggested one request a second as a default. Is that still polite when thirty of us send one at the same second?
- Week 3 called infrastructure that a community governs "ownership". Who pays for the requests we're about to send, and what do we owe them in return?

</details>

## Step 1 — Read a capture's address (10 minutes)

Every capture has an address you can read and rewrite, as the chapter's [Moving Through Time](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-07-archives.html#moving-through-time) shows:

```
https://web.archive.org/web/20040212031928/http://www.thefacebook.com/
```

The 14 digits after `/web/` are the **timestamp**: when the page was captured, in UTC, from year to second. The original address follows them. Take the two apart with [`split()`](https://docs.python.org/3/library/stdtypes.html#str.split) and slices, and read the timestamp with [`pd.to_datetime()`](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html). Then shorten it to a year: the Wayback Machine **redirects** you to the closest capture, and [`response.history`](https://requests.readthedocs.io/en/stable/user/quickstart/#redirection-and-history) keeps the trail.

In [2]:
# CAPTURE: the address above, as a string
CAPTURE = "https://web.archive.org/web/20040212031928/http://www.thefacebook.com/"

In [3]:
# after_web: what follows "/web/" in CAPTURE (split it on "/web/", and take item [1]); print it
after_web = CAPTURE.split("/web/")[1]
print(after_web)

20040212031928/http://www.thefacebook.com/


In [4]:
# stamp: after_web's first 14 characters ([:14]); original: everything from character 15 on ([15:]); print both
stamp = after_web[:14]
original = after_web[15:]
print(stamp, original)

20040212031928 http://www.thefacebook.com/


In [5]:
# when: stamp as a date and time: pd.to_datetime() with format="%Y%m%d%H%M%S" and utc=True; print it
when = pd.to_datetime(stamp, format="%Y%m%d%H%M%S", utc=True)
print(when)

2004-02-12 03:19:28+00:00


In [ ]:
# year_url: "https://web.archive.org/web/2004/thefacebook.com", with a year where the timestamp was
# by_year: request year_url with headers=HEADERS; print its status code and its url, where we landed
year_url = "https://web.archive.org/web/2004/thefacebook.com"
by_year = requests.get(year_url, headers=HEADERS)
print(by_year.status_code, by_year.url)

In [ ]:
# Print by_year.history, the redirects it followed; first: its item [0]; print first.headers["Location"]
print(by_year.history)
first = by_year.history[0]
print(first.headers["Location"])

### ⏱ Microsprint (4 minutes, alone or in pairs)

From the chapter's [Try It: Before They Were Famous](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-07-archives.html#about-this-capture) box.

**Do:** pick an address from the box (`google.com` in 1998, `x.com` in 1999) or a site you knew in middle school, and ask for it with only a year for a timestamp. Print where you landed, and the page's `<title>`. The key asks for `x.com` in 1999.
**Share:** the address, the date of the capture you landed on, and its title.

In [ ]:
# Microsprint: famous: request "https://web.archive.org/web/<year>/<address>" with headers=HEADERS
# Print famous.url; famous_soup: BeautifulSoup(famous.text, "html.parser"); print famous_soup.title
famous = requests.get("https://web.archive.org/web/1999/x.com", headers=HEADERS)
print(famous.url)
famous_soup = BeautifulSoup(famous.text, "html.parser")
print(famous_soup.title)

<details>
<summary>💬 Discuss</summary>

- `CAPTURE` was made at 03:19 UTC on February 12, 2004. What date and time was that at Harvard, where Thefacebook's users were?
- We asked for 2004 and landed on one capture. If you were studying a page in the week before an event, what could "closest" hide from you?
- Week 1's request to Wikipedia had an empty `response.history`. What is each redirect here telling us?

</details>

## Step 2 — The archive's wrapper, and `id_` (7 minutes)

What the Wayback Machine sends back isn't the page as it was captured. It **wraps** the page: it adds its toolbar and scripts, and rewrites the page's links to point back into the archive, at addresses that hold `/web/`. That's convenient for reading and, as the chapter's [Inspecting Archived Pages](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-07-archives.html#inspecting-archived-pages) puts it, poisonous for measurement. Adding `id_` after the timestamp asks for the original capture, unmodified. Count `/web/` in each with [`str.count()`](https://docs.python.org/3/library/stdtypes.html#str.count). How many should the original hold?

In [ ]:
# wrapped: request CAPTURE with headers=HEADERS; print its status code, and wrapped.text.count("/web/")
wrapped = requests.get(CAPTURE, headers=HEADERS)
print(wrapped.status_code, wrapped.text.count("/web/"))

In [6]:
# RAW: CAPTURE with "id_" added after its timestamp: CAPTURE.replace(stamp, stamp + "id_"); print it
RAW = CAPTURE.replace(stamp, stamp + "id_")
print(RAW)

https://web.archive.org/web/20040212031928id_/http://www.thefacebook.com/


In [ ]:
# raw: request RAW the same way; print its status code, and its count of "/web/"
raw = requests.get(RAW, headers=HEADERS)
print(raw.status_code, raw.text.count("/web/"))

In [ ]:
# Given: the two pages we already have, side by side. The for loop runs its indented lines once for each page:
# count its <script> tags, its links (<a> tags with an href), and the links that point into the archive.
# A list in square brackets with "for" inside is a comprehension, a loop written in one line.
# The last line prints the raw page's first three links
for page in [wrapped, raw]:
    page_soup = BeautifulSoup(page.text, "html.parser")
    scripts = page_soup.find_all("script")
    hrefs = [a["href"] for a in page_soup.find_all("a", href=True)]
    into_archive = [href for href in hrefs if "/web/" in href]
    print(len(scripts), "scripts,", len(hrefs), "links,", len(into_archive), "into the archive")
print(hrefs[:3])

<details>
<summary>💬 Discuss</summary>

- Suppose you counted `<script>` tags on wrapped captures to study how JavaScript spread after 2004. What would you have found, and would anything have warned you?
- In week 5, the Inspector showed the page after scripts ran and View Source showed what the server sent. Which is `RAW` closer to, and which does your browser show you for a capture?
- Look at the raw page's first three links. If one carries a `PHPSESSID`, a session's ID, what does that tell you about the crawler, and about what a logged-in student in 2004 would have seen instead?

</details>

## Step 3 — The Availability API: a question we can answer (12 minutes)

The [Availability API](https://archive.org/help/wayback_api.php) answers one question in JSON: which capture of this address is closest to this date? Pass the address and the date as a [`params`](https://requests.readthedocs.io/en/stable/user/quickstart/#passing-parameters-in-urls) dictionary; `requests` encodes them. Start with a question whose answer we know: Thefacebook on February 12, 2004. Which capture should it name? Print what came back before reading it, then go down the JSON one level a cell, as the chapter's [The Availability API](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-07-archives.html#the-availability-api) does. Last, ask about an address nobody has archived.

In [7]:
# avail_url: "https://archive.org/wayback/available"
# facebook_params: a dictionary with "url": "thefacebook.com" and "timestamp": "20040212"
avail_url = "https://archive.org/wayback/available"
facebook_params = {"url": "thefacebook.com", "timestamp": "20040212"}

In [8]:
# facebook: request avail_url with params=facebook_params and headers=HEADERS
# Print its status code and the first 300 characters of its text
facebook = requests.get(avail_url, params=facebook_params, headers=HEADERS)
print(facebook.status_code, facebook.text[:300])

200 {"url": "thefacebook.com", "archived_snapshots": {"closest": {"status": "200", "available": true, "url": "http://web.archive.org/web/20040212031928/http://www.thefacebook.com:80/", "timestamp": "20040212031928"}}, "timestamp": "20040212"}


In [9]:
# facebook_data: the response's .json(); show its keys
facebook_data = facebook.json()
facebook_data.keys()

dict_keys(['url', 'archived_snapshots', 'timestamp'])

In [10]:
# Show facebook_data["archived_snapshots"]
facebook_data["archived_snapshots"]

{'closest': {'status': '200',
  'available': True,
  'url': 'http://web.archive.org/web/20040212031928/http://www.thefacebook.com:80/',
  'timestamp': '20040212031928'}}

In [11]:
# closest: facebook_data["archived_snapshots"]["closest"]; print its "timestamp" and its "url"
closest = facebook_data["archived_snapshots"]["closest"]
print(closest["timestamp"], closest["url"])

20040212031928 http://web.archive.org/web/20040212031928/http://www.thefacebook.com:80/


In [12]:
# nowhere_params: the same, with "url": "this-site-does-not-exist-4617.org"; nowhere: send them
# Print its status code and its text
nowhere_params = {"url": "this-site-does-not-exist-4617.org", "timestamp": "20040212"}
nowhere = requests.get(avail_url, params=nowhere_params, headers=HEADERS)
print(nowhere.status_code, nowhere.text)

200 {"url": "this-site-does-not-exist-4617.org", "archived_snapshots": {}, "timestamp": "20040212"}


### ⏱ Microsprint (3 minutes, alone or in pairs)

What does the API do with a parameter it doesn't know?

**Do:** ask about Thefacebook again, with `"timestamp"` misspelled as `"timestmp"`, and print the text that comes back.
**Share:** the date of the capture it named, and whether anything told you about the misspelling.

In [13]:
# Microsprint: typo_params: "url": "thefacebook.com" and "timestmp": "20040212" (misspelled on purpose)
# typo: send them to avail_url; print its status code and its text
typo_params = {"url": "thefacebook.com", "timestmp": "20040212"}
typo = requests.get(avail_url, params=typo_params, headers=HEADERS)
print(typo.status_code, typo.text)

200 {"url": "thefacebook.com", "archived_snapshots": {"closest": {"status": "200", "available": true, "url": "http://web.archive.org/web/20080725005455/http://www.thefacebook.com/", "timestamp": "20080725005455"}}}


<details>
<summary>💬 Discuss</summary>

- Is `closest`'s timestamp the one we took apart in Step 1? Its `url` differs from `CAPTURE` in two small ways: which, and do they matter?
- `nowhere` came back with status 200 and an empty `archived_snapshots`. What would `["closest"]` do on that answer, and would its error tell you why? The chapter checks for a capture first, and we don't.
- The misspelled `timestmp` drew no error: the API's page says that without a timestamp it returns the most recent capture. In a loop over a hundred dates, how would you notice?
- On 2026-11-17, after about 35 requests in ten minutes, this API answered the key's machine `429 Too Many Requests`, with a page of HTML. Which of our lines would stop on that answer, and which line would have shown you the reason?

</details>

# Part 2 — Wednesday: one capture at a time, then every capture

Monday's Availability API finds one capture near a date. We use it on CNN's home page in three eras: once by hand, then in a given loop. Then the **CDX Server** lists every capture of a page in one answer, with what the crawler was told each time: how often Facebook's terms were saved, how big each version was, and what `facebook.com` answered in 2005.

## Step 4 — Set up again (3 minutes)

Part 2 starts fresh, so a student who missed Monday can start here: nothing from Monday's cells is needed. Put your email in the given cell and run it. It imports what we need, and sets Monday's `avail_url` and `checkpoints`, the address of the saved copies to load if the Archive refuses a request.

In [1]:
# Given: imports; your User-Agent (put your email in it); Monday's Availability API; the saved copies' address
import time
import requests
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

HEADERS = {"User-Agent": "WebDataScience/1.0 (your-email@colorado.edu)"}
avail_url = "https://archive.org/wayback/available"
checkpoints = "https://raw.githubusercontent.com/cuinfoscience/INFO4617-Fall2026/main/labs/week-07/data/"

<details>
<summary>💬 Discuss</summary>

- On Monday, did the Archive answer every request you sent? If one stopped, what did its error say, and did the error name the cause?

</details>

## Step 5 — CNN in 2000 by hand, then three eras (12 minutes)

Monday's moves, on a new case: ask the Availability API for CNN's home page on August 15, 2000, add `id_` to the capture it names, fetch it, and count its links with BeautifulSoup (week 6). The given [`get_snapshot_stats(url, timestamp)`](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-07-archives.html#comparing-multiple-snapshots) is the chapter's **function**: a name for those lines, which runs them for any address and date, and hands back four counts. A given loop runs it for 2000, 2012 and 2024, pausing a second between dates with [`time.sleep(1)`](https://docs.python.org/3/library/time.html#time.sleep).

In [2]:
# cnn_params: "url": "www.cnn.com" and "timestamp": "20000815"; cnn: send them to avail_url with headers=HEADERS
# Print its status code and the first 300 characters of its text. An "error" in it? Wait a minute, and run it again
cnn_params = {"url": "www.cnn.com", "timestamp": "20000815"}
cnn = requests.get(avail_url, params=cnn_params, headers=HEADERS)
print(cnn.status_code, cnn.text[:300])

200 {"url": "www.cnn.com", "archived_snapshots": {"closest": {"status": "200", "available": true, "url": "http://web.archive.org/web/20000815052826/http://www.cnn.com:80/", "timestamp": "20000815052826"}}, "timestamp": "20000815"}


In [3]:
# cnn_data: cnn's .json(); cnn_closest: cnn_data["archived_snapshots"]["closest"]; cnn_stamp: its "timestamp"
cnn_data = cnn.json()
cnn_closest = cnn_data["archived_snapshots"]["closest"]
cnn_stamp = cnn_closest["timestamp"]

In [4]:
# cnn_raw: cnn_closest["url"] with "id_" after its timestamp (replace cnn_stamp with cnn_stamp + "id_"); print it
cnn_raw = cnn_closest["url"].replace(cnn_stamp, cnn_stamp + "id_")
print(cnn_raw)

http://web.archive.org/web/20000815052826id_/http://www.cnn.com:80/


In [ ]:
# cnn_page: request cnn_raw with headers=HEADERS; cnn_soup: BeautifulSoup(cnn_page.text, "html.parser")
# Print how many <a> tags it has: len(cnn_soup.find_all("a"))
cnn_page = requests.get(cnn_raw, headers=HEADERS)
cnn_soup = BeautifulSoup(cnn_page.text, "html.parser")
print(len(cnn_soup.find_all("a")))

In [5]:
# Given: the chapter's get_snapshot_stats(): the cells above as a function, without the chapter's try/except.
# It returns the year we asked for, the capture's own timestamp, and four counts from the raw capture
def get_snapshot_stats(url, timestamp):
    response = requests.get(avail_url, params={"url": url, "timestamp": timestamp}, headers=HEADERS)
    data = response.json()
    snapshot = data["archived_snapshots"]["closest"]
    raw_url = snapshot["url"].replace(snapshot["timestamp"], snapshot["timestamp"] + "id_")
    page = requests.get(raw_url, headers=HEADERS)
    soup = BeautifulSoup(page.text, "html.parser")
    return {
        "year": timestamp[:4],
        "capture": snapshot["timestamp"],
        "num_links": len(soup.find_all("a")),
        "text_length": len(soup.get_text()),
        "num_images": len(soup.find_all("img")),
        "num_scripts": len(soup.find_all("script")),
    }

In [ ]:
# Given: CNN in August 2000, 2012 and 2024, with a second's pause between dates; comparison: a row for each
stats = []
for ts in ["20000815", "20120815", "20240815"]:
    row = get_snapshot_stats("www.cnn.com", ts)
    stats.append(row)
    time.sleep(1)
comparison = pd.DataFrame(stats)
comparison

In [6]:
# Given, only if the cell above stopped: remove the "#" from the next line, and run this cell.
# It loads the table the key's code saved (labs/week-07/data/README.md says when)
# comparison = pd.read_csv(checkpoints + "cnn-three-eras.csv", dtype={"year": str, "capture": str})

### ⏱ Microsprint (4 minutes, alone or in pairs)

A page-statistics version of Additional Exercise 2, which compares two competing organizations.

**Do:** run `get_snapshot_stats()` for a competitor of CNN's (`www.nytimes.com`, `www.foxnews.com`, `www.usatoday.com`) in August 2000, and again in August 2024. The key asks for `www.nytimes.com`.
**Share:** the site, the two captures you got, and its numbers of links and scripts in each.

In [ ]:
# Microsprint: print get_snapshot_stats() for another news site at "20000815"; time.sleep(1); then at "20240815"
print(get_snapshot_stats("www.nytimes.com", "20000815"))
time.sleep(1)
print(get_snapshot_stats("www.nytimes.com", "20240815"))

<details>
<summary>💬 Discuss</summary>

- The `capture` column holds each capture's own timestamp. Asked on 2026-11-17 for `www.nytimes.com` on August 15, 2000, the API named a capture from November 9, 2000. How far from August 15 did your captures land, and when would that matter?
- Which count grew most from 2000 to 2024? Week 6's "when 'static' isn't" warned about pages built by scripts: what can `requests` not see on the 2024 page?
- On 2026-11-17 the API answered this request, for a while, with status 200, an empty `archived_snapshots`, and an `"error"` whose code was `timeout`. Which line of `get_snapshot_stats()` stops on that answer, and how could you tell "never archived" from "ask again"?

</details>

## Step 6 — The CDX Server: three captures first (8 minutes)

The Availability API names one capture. The [CDX Server API](https://github.com/internetarchive/wayback/tree/master/wayback-cdx-server) lists them all, a row for each, oldest first. Before a whole history, a small request: Thefacebook's first three captures ([`limit=3`](https://github.com/internetarchive/wayback/tree/master/wayback-cdx-server#query-result-limits)), with two fields ([`fl`](https://github.com/internetarchive/wayback/tree/master/wayback-cdx-server#field-order)): each one's timestamp, and the [status code](https://developer.mozilla.org/en-US/docs/Web/HTTP/Reference/Status) (week 5) the crawler received. [`output=json`](https://github.com/internetarchive/wayback/tree/master/wayback-cdx-server#output-format-json) asks for a list of lists. How many rows should come back, and from which year?

In [7]:
# cdx_url: "https://web.archive.org/cdx/search/cdx"
# small_params: "url": "thefacebook.com", "output": "json", "fl": "timestamp,statuscode", and "limit": 3
cdx_url = "https://web.archive.org/cdx/search/cdx"
small_params = {"url": "thefacebook.com", "output": "json",
                "fl": "timestamp,statuscode", "limit": 3}

In [ ]:
# small: request cdx_url with params=small_params and headers=HEADERS
# Print its status code and the first 300 characters of its text
small = requests.get(cdx_url, params=small_params, headers=HEADERS)
print(small.status_code, small.text[:300])

In [ ]:
# small_rows: the response's .json(); print how many rows it has, then row 0 and row 1
small_rows = small.json()
print(len(small_rows))
print(small_rows[0], small_rows[1])

<details>
<summary>💬 Discuss</summary>

- Row 0 isn't a capture. What is it, and why does that make `len(small_rows)` one more than `limit`?
- Week 4's JSON often came as a list of dictionaries. Each row here is a list. What do you lose, and what does the Archive save, across millions of rows?
- Leave out `output` and the same rows come back as plain text. Which line would stop then, and what would the line before it have shown you?

</details>

## Step 7 — Every version of Facebook's terms (10 minutes)

Now [the chapter's query](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-07-archives.html#the-cdx-server-api), for `facebook.com/terms`. The given `terms_params` asks for four fields, up to a fixed day, `to=20260930`, so that a rerun next year gives the same rows. Add the two parameters that do the work on the Archive's side. [`filter`](https://github.com/internetarchive/wayback/tree/master/wayback-cdx-server#filtering) set to `statuscode:200` keeps the captures that succeeded. [`collapse`](https://github.com/internetarchive/wayback/tree/master/wayback-cdx-server#collapsing) set to `digest` drops each capture whose **digest**, a fingerprint of its content, matches the one before it. What's left is a row for each time the page changed.

In [8]:
# Given: the chapter's query, with an end date ("to") so that a rerun gives the same rows
terms_params = {
    "url": "facebook.com/terms",
    "output": "json",
    "fl": "timestamp,original,statuscode,length",
    "to": "20260930",
}

In [9]:
# Add two keys to terms_params: "filter", set to "statuscode:200", and "collapse", set to "digest"
terms_params["filter"] = "statuscode:200"
terms_params["collapse"] = "digest"

In [ ]:
# terms: request cdx_url with params=terms_params and headers=HEADERS
# Print its status code and the first 300 characters of its text
terms = requests.get(cdx_url, params=terms_params, headers=HEADERS)
print(terms.status_code, terms.text[:300])

In [ ]:
# terms_rows: the response's .json(); df: pd.DataFrame() of terms_rows[1:], with columns=terms_rows[0]
# Print how many rows df has
terms_rows = terms.json()
df = pd.DataFrame(terms_rows[1:], columns=terms_rows[0])
print(len(df), "versions")

In [10]:
# Given, only if the cells above stopped: remove the "#" from the next two lines, and run this cell.
# It loads the answer the key's code saved (labs/week-07/data/README.md says when)
# terms_rows = requests.get(checkpoints + "cdx-facebook-terms.json", headers=HEADERS).json()
# df = pd.DataFrame(terms_rows[1:], columns=terms_rows[0])

In [ ]:
# Make df["timestamp"] dates: pd.to_datetime() with format="%Y%m%d%H%M%S"
# Make df["length"] numbers: pd.to_numeric() with errors="coerce". Print the first and last timestamp
df["timestamp"] = pd.to_datetime(df["timestamp"], format="%Y%m%d%H%M%S")
df["length"] = pd.to_numeric(df["length"], errors="coerce")
print(df["timestamp"].min(), df["timestamp"].max())

<details>
<summary>💬 Discuss</summary>

- How many requests did that take? How many would the Availability API need to tell us the same thing?
- `collapse=digest` drops a capture only when it matches the one just before it. If Facebook put an old wording back, would that version be counted again?
- If the CDX Server answered `429`, or dropped the connection as it sometimes does, which line would stop, and would its error tell you why?

</details>

## Step 8 — Plot the archival record (7 minutes)

Two questions for the record: how often was a new version captured, and how big was each? Make the timestamp the index, so that [`resample()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html) can group the rows by quarter (`"QS"`, each quarter's start). A given cell counts each quarter's captures and averages their `length`, as the chapter's [Filtering and Analyzing CDX Data](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-07-archives.html#filtering-and-analyzing-cdx-data) does, and another plots both. `length` is the size of the archived record, compressed: good for spotting jumps, not for saying how big the page was.

In [ ]:
# df_ok: df with "timestamp" as its index: df.set_index("timestamp")
df_ok = df.set_index("timestamp")

In [ ]:
# Given: a row for each quarter: captures, how many versions were captured in it, and mean_size, their mean length
quarterly = df_ok.resample("QS").agg(
    captures=("length", "size"),
    mean_size=("length", "mean"),
)

In [ ]:
# Print the three quarters with the most captures: quarterly.nlargest(3, "captures")
print(quarterly.nlargest(3, "captures"))

In [ ]:
# Given: plot both, one above the other, on one time axis
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
quarterly["captures"].plot(ax=axes[0])
axes[0].set_title("facebook.com/terms: distinct captures (status 200, collapsed on digest)")
axes[0].set_ylabel("Captures per quarter")
quarterly["mean_size"].plot(ax=axes[1])
axes[1].set_title("facebook.com/terms: mean size of the archived record")
axes[1].set_ylabel("Bytes, compressed")
axes[1].set_xlabel("Quarter")
fig.tight_layout()
plt.show()

<details>
<summary>💬 Discuss</summary>

- With `collapse=digest`, every row is a change in the page's bytes. Is the busiest quarter the one in which Facebook changed its terms most, or the one in which the Archive paid most attention?
- Pick a sudden jump in size. What would you open to tell a redesign from a move behind a login?
- In week 1 we plotted an article's daily views. Whose behavior does this plot measure: Facebook's, its readers', or the Archive's?

</details>

## Step 9 — What the calendar's colors mean (11 minutes)

The Wayback calendar colors each day by the status code the crawler received (week 5): blue for 2xx, green for 3xx, orange for 4xx, red for 5xx. The CDX Server reports the same codes, so leave out the filter and look at what Step 7 dropped: every capture of `facebook.com` in 2005, the year the address changed hands ([What the Colors Mean](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-07-archives.html#what-the-colors-mean)). [`pd.crosstab()`](https://pandas.pydata.org/docs/reference/api/pandas.crosstab.html) counts the captures for each month and code. A given function repeats it for any page.

In [11]:
# Given: every capture of facebook.com in 2005 ("from" and "to"), with no filter and no collapse
fb2005_params = {
    "url": "facebook.com",
    "output": "json",
    "fl": "timestamp,statuscode",
    "from": "2005",
    "to": "2005",
}

In [ ]:
# fb2005: request cdx_url with params=fb2005_params and headers=HEADERS; print its status and text[:300]
fb2005 = requests.get(cdx_url, params=fb2005_params, headers=HEADERS)
print(fb2005.status_code, fb2005.text[:300])

In [ ]:
# fb2005_rows: the response's .json(); fb2005_df: a DataFrame of fb2005_rows[1:], with columns=fb2005_rows[0]
# Print how many rows it has
fb2005_rows = fb2005.json()
fb2005_df = pd.DataFrame(fb2005_rows[1:], columns=fb2005_rows[0])
print(len(fb2005_df), "captures")

In [12]:
# Given, only if the cells above stopped: remove the "#" from the next two lines, and run this cell.
# It loads the answer the key's code saved (labs/week-07/data/README.md says when)
# fb2005_rows = requests.get(checkpoints + "cdx-facebook-2005.json", headers=HEADERS).json()
# fb2005_df = pd.DataFrame(fb2005_rows[1:], columns=fb2005_rows[0])

In [ ]:
# fb2005_df["month"]: the timestamp's first 6 characters, fb2005_df["timestamp"].str[:6]
# counts: pd.crosstab() of fb2005_df["month"] and fb2005_df["statuscode"]; print it
fb2005_df["month"] = fb2005_df["timestamp"].str[:6]
counts = pd.crosstab(fb2005_df["month"], fb2005_df["statuscode"])
print(counts)

In [13]:
# Given: the cells above as a function, by year: every capture of url from one year to another, with no filter,
# counted by year (the timestamp's first 4 characters) and status code
def status_by_year(url, start, end):
    params = {"url": url, "output": "json", "fl": "timestamp,statuscode", "from": start, "to": end}
    rows = requests.get(cdx_url, params=params, headers=HEADERS).json()
    captures = pd.DataFrame(rows[1:], columns=rows[0])
    return pd.crosstab(captures["timestamp"].str[:4], captures["statuscode"])

### ⏱ Microsprint (4 minutes, alone or in pairs)

A status-code version of Additional Exercise 4, on a page the chapter's EDGI study watched: the EPA's climate change page.

**Do:** run `status_by_year()` for `"epa.gov/climatechange"`, from `"2016"` to `"2018"`.
**Share:** the year its captures stop being 200s, and the status codes that replaced them.

In [ ]:
# Microsprint: status_by_year("epa.gov/climatechange", "2016", "2018")
status_by_year("epa.gov/climatechange", "2016", "2018")

<details>
<summary>💬 Discuss</summary>

- The Archive recorded each refusal accurately. For which research question are the 403 rows noise, and for which are they the finding?
- A soft 404 is a "Page not found" page sent with a 200. Would Step 7's filter keep it or drop it, and how would you catch one?
- In week 3's dead-endpoint forensics, the way an endpoint failed was the finding. What does the status column tell us here that the pages themselves can't?
- The lab keeps a saved copy of each CDX answer in case the Archive refuses yours. What can a saved copy not tell you that the live request can?

</details>

# Part 3 — Friday: ten versions of the terms, and how much each changed

Wednesday's sizes say that the terms changed, not how. For that we need the words. We fetch one version by hand, then ten, two years apart, with a given function that saves each one in a file. Then a **bag of words** turns a text into word counts that we can compare: first on three short sentences, then on the terms.

## Step 10 — Set up again (2 minutes)

Part 3 starts fresh: nothing from Monday's or Wednesday's cells is needed. Put your email in the given cell and run it. It imports what we need, [gensim](https://radimrehurek.com/gensim/) included (the opening cell says what to do if that line stops), and sets Monday's `avail_url` and the address of the saved copies.

In [1]:
# Given: imports; your User-Agent (put your email in it); Monday's Availability API; the saved copies' address
import os
import json
import time
import requests
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from gensim.utils import simple_preprocess
from gensim import corpora, similarities

HEADERS = {"User-Agent": "WebDataScience/1.0 (your-email@colorado.edu)"}
avail_url = "https://archive.org/wayback/available"
checkpoints = "https://raw.githubusercontent.com/cuinfoscience/INFO4617-Fall2026/main/labs/week-07/data/"

<details>
<summary>💬 Discuss</summary>

- On Wednesday, the CDX Server sent every version's size in one answer. Why can't one request bring back every version's words?

</details>

## Step 11 — One version's text, then ten (13 minutes)

Monday's moves once more: ask the Availability API for `facebook.com/terms` at the start of 2010, fetch the capture it names with `id_`, and keep the page's words with BeautifulSoup's [`get_text(separator=" ", strip=True)`](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#get-text). The given [`get_wayback_content(url, date)`](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-07-archives.html#retrieving-historical-content) is the chapter's function for those lines. A given loop runs it for the start of every other year from 2006 to 2024, and saves each result in `wayback_cache/` with [`json.dump()`](https://docs.python.org/3/library/json.html#json.dump), as in the chapter's [Caching Raw Captures](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-07-archives.html#caching-raw-captures). It skips a date whose file is there, so a second run asks the Archive for nothing.

In [2]:
# tos_params: "url": "facebook.com/terms" and "timestamp": "20100101"; tos: send them to avail_url with headers=HEADERS
tos_params = {"url": "facebook.com/terms", "timestamp": "20100101"}
tos = requests.get(avail_url, params=tos_params, headers=HEADERS)

In [3]:
# tos_data: tos's .json(); tos_closest: tos_data["archived_snapshots"]["closest"]; print its "timestamp"
tos_data = tos.json()
tos_closest = tos_data["archived_snapshots"]["closest"]
print(tos_closest["timestamp"])

20101006200541


In [4]:
# tos_stamp: tos_closest["timestamp"]; tos_raw: tos_closest["url"] with "id_" after tos_stamp
tos_stamp = tos_closest["timestamp"]
tos_raw = tos_closest["url"].replace(tos_stamp, tos_stamp + "id_")

In [ ]:
# tos_page: request tos_raw; tos_soup: BeautifulSoup(tos_page.text, "html.parser")
# tos_text: tos_soup.get_text(separator=" ", strip=True); print how many words (len(tos_text.split())), and tos_text[:300]
tos_page = requests.get(tos_raw, headers=HEADERS)
tos_soup = BeautifulSoup(tos_page.text, "html.parser")
tos_text = tos_soup.get_text(separator=" ", strip=True)
print(len(tos_text.split()), "words:", tos_text[:300])

In [5]:
# Given: the chapter's get_wayback_content(): the cells above as a function, without the chapter's try/except.
# date.strftime("%Y%m%d") writes a date as the API's timestamp. It returns the capture's own timestamp, its id_
# address, its number of words, and its text
def get_wayback_content(url, date):
    timestamp = date.strftime("%Y%m%d")
    response = requests.get(avail_url, params={"url": url, "timestamp": timestamp}, headers=HEADERS)
    data = response.json()
    snapshot = data["archived_snapshots"]["closest"]
    raw_url = snapshot["url"].replace(snapshot["timestamp"], snapshot["timestamp"] + "id_")
    page = requests.get(raw_url, headers=HEADERS)
    soup = BeautifulSoup(page.text, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    return {
        "timestamp": snapshot["timestamp"],
        "url": raw_url,
        "word_count": len(text.split()),
        "content": text,
    }

In [ ]:
# Given: the start of every other year, 2006 to 2024 ("2YS"). For each date with no file in wayback_cache/ yet:
# get that version, save it there under the date's name, and pause a second. Then list the folder.
# If it stops, wait a minute and run it again: it goes on from the first date that has no file
os.makedirs("wayback_cache", exist_ok=True)
dates = pd.date_range("2006-01-01", "2024-01-01", freq="2YS")
for date in dates:
    name = date.strftime("%Y%m%d")
    cache_path = f"wayback_cache/{name}.json"
    if not os.path.exists(cache_path):
        result = get_wayback_content("facebook.com/terms", date)
        with open(cache_path, "w") as f:
            json.dump(result, f)
        time.sleep(1)
print(sorted(os.listdir("wayback_cache")))

In [ ]:
# Given: read the ten files back, in date order, into results, a list of dictionaries
results = []
for date in dates:
    name = date.strftime("%Y%m%d")
    with open(f"wayback_cache/{name}.json") as f:
        results.append(json.load(f))

In [6]:
# Given, only if the cells above stopped: remove the "#" from the next line, and run this cell.
# It loads the ten versions the key's code saved (labs/week-07/data/README.md says when)
# results = requests.get(checkpoints + "facebook-terms-versions.json", headers=HEADERS).json()

In [ ]:
# Given: the date we asked for, then the capture we got and its number of words, for each of the ten
for i in range(len(results)):
    version = results[i]
    print(dates[i].date(), "->", version["timestamp"], version["word_count"], "words")

<details>
<summary>💬 Discuss</summary>

- Asked on 2026-11-17, the Availability API named one capture, from October 6, 2010, for all four of our dates from 2006 to 2012. Why might a page have no capture near 2006 at this address, and what will four copies of one version do to Step 13?
- The cache names each file by its date alone. What happens if you run the loop for `twitter.com/tos` on the same dates?
- `get_text()` keeps every word on the page, menus and footers included. What will that do to Step 13's comparisons?
- Week 2 called terms of service "quasi-law". Who besides researchers would want every past version of one, and why?

</details>

## Step 12 — A bag of words, on three sentences (9 minutes)

A **bag of words** describes a text by how often each word appears in it, in any order. Try it on three sentences before the terms: two alike, one not. gensim's [`simple_preprocess()`](https://radimrehurek.com/gensim/utils.html#gensim.utils.simple_preprocess) splits a text into lowercase **tokens**. A given cell drops the book's 198 **stopwords**, words like *your*, too common to tell texts apart. A [`Dictionary`](https://radimrehurek.com/gensim/corpora/dictionary.html) numbers the words, and `doc2bow()` counts them. A [`SparseMatrixSimilarity`](https://radimrehurek.com/gensim/similarities/docsim.html) index gives two bags' **cosine similarity**: 1 for the same words in the same proportions, 0 for none in common.

In [7]:
# tokens: simple_preprocess() of the sentence "We may share your data with Facebook's partners."; print it
tokens = simple_preprocess("We may share your data with Facebook's partners.")
print(tokens)

['we', 'may', 'share', 'your', 'data', 'with', 'facebook', 'partners']


In [8]:
# sentences: a list of three: "We collect your data.", "We collect and share your data.", "Cats sleep all day."
sentences = [
    "We collect your data.",
    "We collect and share your data.",
    "Cats sleep all day.",
]

In [9]:
# Given: the book's 198 stopwords, from its repository; the chapter's preprocess_document(): a text's tokens without
# the stopwords (a comprehension: a loop inside the brackets); and tiny_docs, each sentence's tokens
book_data = "https://raw.githubusercontent.com/cuinfoscience/Web-Data-Science-Book/main/data/"
stop_words = set(requests.get(book_data + "stopwords-en.txt", headers=HEADERS).text.split())

def preprocess_document(text):
    tokens = simple_preprocess(text)
    return [token for token in tokens if token not in stop_words]

tiny_docs = [preprocess_document(sentence) for sentence in sentences]
print(len(stop_words), "stopwords")
print(tiny_docs)

198 stopwords
[['collect', 'data'], ['collect', 'share', 'data'], ['cats', 'sleep', 'day']]


In [10]:
# tiny_dictionary: corpora.Dictionary(tiny_docs); print its token2id, the number it gave each word
tiny_dictionary = corpora.Dictionary(tiny_docs)
print(tiny_dictionary.token2id)

{'collect': 0, 'data': 1, 'share': 2, 'cats': 3, 'day': 4, 'sleep': 5}


In [11]:
# Print sentence 1's bag of words: tiny_dictionary.doc2bow(tiny_docs[1])
print(tiny_dictionary.doc2bow(tiny_docs[1]))

[(0, 1), (1, 1), (2, 1)]


In [12]:
# Given: every sentence's bag, and an index that compares one bag with all three
tiny_corpus = [tiny_dictionary.doc2bow(doc) for doc in tiny_docs]
tiny_index = similarities.SparseMatrixSimilarity(tiny_corpus, num_features=len(tiny_dictionary))

In [13]:
# scores: sentence 0's similarity to all three, tiny_index[tiny_corpus[0]]; print scores.round(3)
scores = tiny_index[tiny_corpus[0]]
print(scores.round(3))

[1.    0.816 0.   ]


### ⏱ Microsprint (3 minutes, alone or in pairs)

*not* is on the stopword list. What does that do to a sentence that turns on it?

**Do:** print `preprocess_document()` of "We do not share your data.", and then of "We share your data."
**Share:** the two lists of tokens, and what their similarity would be.

In [14]:
# Microsprint: print preprocess_document("We do not share your data."); then of "We share your data."
print(preprocess_document("We do not share your data."))
print(preprocess_document("We share your data."))

['share', 'data']
['share', 'data']


<details>
<summary>💬 Discuss</summary>

- Sentences 0 and 1 scored 0.816, not 1. Which word made the difference, and which words made none?
- The microsprint's two sentences left the same tokens. What would our measure say about a version of the terms that added one *not*, and how else could you catch that change?
- Week 6's parsers kept one table or one infobox; this keeps every word on a page. When is each the right choice?

</details>

## Step 13 — How much each version changed (8 minutes)

Now the ten versions. A given cell runs `preprocess_document()` on each, and builds one `Dictionary`, corpus and index for all ten, as the chapter's [Visualizing Document Change Over Time](https://cuinfoscience.github.io/Web-Data-Science-Book/ch-07-archives.html#visualizing-document-change-over-time) does. One minus the similarity of a version and the one before it is its **change score**: 0 for no change, 1 for no words in common. Work out one score by hand. A given cell works out the rest and plots each against the date of the later capture, so that a spike can be matched with an event.

In [ ]:
# Given: every version's tokens; one Dictionary, corpus and index for all ten, as in Step 12
processed_docs = [preprocess_document(version["content"]) for version in results]
dictionary = corpora.Dictionary(processed_docs)
corpus = [dictionary.doc2bow(doc) for doc in processed_docs]
index = similarities.SparseMatrixSimilarity(corpus, num_features=len(dictionary))
print(len(corpus), "versions,", len(dictionary), "different words")

In [ ]:
# Print the change from version 0 to version 1, rounded to 3 places: round(1 - index[corpus[0]][1], 3)
# Then the change from version 0 to the last version, [9]
print(round(1 - index[corpus[0]][1], 3))
print(round(1 - index[corpus[0]][9], 3))

In [ ]:
# Given: each version's change from the one before it, and the date of the later capture; plot and print them
change_scores = [1 - index[corpus[i - 1]][i] for i in range(1, len(corpus))]
stamps = [version["timestamp"] for version in results[1:]]
when = pd.to_datetime(stamps, format="%Y%m%d%H%M%S")
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(when, change_scores, marker="o")
ax.set_title("facebook.com/terms: change from the version before")
ax.set_xlabel("Date of the later capture")
ax.set_ylabel("Change score (0 = identical)")
plt.show()
print(pd.Series(change_scores, index=when.date).round(3))

### ⏱ Microsprint (4 minutes, alone or in pairs)

A short version of Additional Exercise 3, which compares every version with every other.

**Do:** pick one version, from 0 to 9, and print its similarity to all ten, rounded to 2 places. The key picks version 4.
**Share:** your version's capture date, the version it is most like other than itself, and the one it is least like.

In [ ]:
# Microsprint: print(index[corpus[4]].round(2)), with your version's number in place of 4
print(index[corpus[4]].round(2))

<details>
<summary>💬 Discuss</summary>

- Pick the biggest spike. What happened at Facebook around then, and how would you check it with archived sources rather than from memory?
- Captures two years apart can't see a change that came and went between them. What would the chapter's quarterly dates add, and what would they cost the Archive?
- The chapter's graduate exercise warns that observed change is confounded with attention. Where could that confound enter this plot?

</details>

## Step 14 — What did we learn? (3 minutes)

In a few sentences, as comments:

- How many distinct versions of `facebook.com/terms` did the CDX Server list in Step 7, and how many different captures did our ten dates reach in Step 11? What does the gap mean for Step 13's timeline?
- In Step 9, what share of `facebook.com`'s captures in 2005 weren't 200s? What would a study that kept only the 200s have missed?
- Steps 3, 6 and 12 each began with a small request whose answer we could predict. What did one of them show you that the full run would have hidden?
- What would you write in a methods section about how the Wayback Machine shaped these results?

**Recommended Exercises ahead** (in the chapter's companion notebook, due Sunday):

- Step 1, query the CDX Server for your page's history, and handle a 429: Steps 6–7's query; the waiting and retrying is yours to write (Steps 3 and 7's Discuss questions).
- Step 2, build the capture DataFrame and print its date range and size: Step 7.
- Step 3, plot captures per year and page size over time: Step 8's `resample()` and two-panel plot.
- Step 4, fetch the earliest and latest captures with `id_` and cache them: Step 2's `id_`, and Step 11's cache.
- Step 5, extract the text of both captures and measure their similarity: Step 11's `get_text()`, and Steps 12–13.
- Step 6, interpret: Steps 8, 9 and 13, and the Discuss questions about gaps and capture bias.

In [15]:
# Your answers here, as comments
# PLACEHOLDER: ANSWERS 1 AND 2 NEED A LIVE RUN (see the key note). web.archive.org refused this key's machine,
#   so Steps 5-9, 11 and 13 have no outputs to answer from. What is known without it:
# 1. [Step 7: rows in df] distinct versions. Our ten dates reached only 7 different captures: asked on
#    2026-11-17, the Availability API named the capture of October 6, 2010 for 2006, 2008, 2010 and 2012 alike.
#    So three of Step 13's nine change scores compare a version with itself and are 0, and the timeline can say
#    nothing about 2006-2010. [Check against Step 11's last cell.]
# 2. [Step 9: captures that weren't 200s / all captures] of facebook.com's captures in 2005. The chapter's calendar
#    shows 403s from April 10 to early August, while the address changed hands: a study of 200s alone would have
#    seen a gap, not a refusal.
# 3. Step 3's: the capture the API named for Thefacebook was the one we took apart in Step 1, 20040212031928, so
#    "closest" meant what we thought. Then the misspelled "timestmp" drew no error and a capture from 2008. In a
#    loop over ten dates, ten copies of the latest capture would have looked like ten answers. And Step 12's
#    three sentences scored 1, 0.816 and 0: the score counts shared words, and a sentence with "not" added
#    scores 1, because "not" is a stopword.
# 4. That the Archive's attention, not only the page, set which versions we saw: the crawl's schedule,
#    collapse=digest, and the capture "closest" to each date, which was up to four years away. And who could
#    collect at all: web.archive.org refused every request from this key's cloud machine on 2026-10-21 and on
#    2026-11-17, so where a study runs from is part of its method.

## Friday: from the lab to the textbook (15 minutes)

What did the lab turn up that the chapter should know? Some places to look:

- Step 5's table, next to the chapter's illustrative one in "Comparing Multiple Snapshots". Should the chapter print real output, with its date?
- Step 11's dates. The chapter's loop asks for `facebook.com/terms` every quarter from 2005; ours got one capture from 2010 for four dates. Should `get_wayback_content()` report how far each capture is from the date asked for?
- Step 11's cache, which names files by date alone (the Discuss question). Is that a bug in the chapter's code, and what would you name the files instead?
- Step 3's misspelled `timestmp`, which the API answered without an error. Does the chapter's "Common Issues to Debug" warn about it?
- The chapter wraps every request in `try`/`except` with `raise_for_status()`; we left them out. After this week, which guard would you keep, and should the chapter say why it's there?
- Step 8's `length`, which the chapter calls "page size". Is "compressed record size" clearer, or just more words?
- A cell that failed this week, and what the chapter's "Common Issues to Debug" should say about it.
- A microsprint that would make a good exercise.

[`handouts/common/revision-framework.md`](../../handouts/common/revision-framework.md) says how to propose a revision.